# Day 84: Entropy Balancing & Calibration Weights

*Beginner to Ninja: Math for Causal Inference*

Instead of estimating a propensity score and hoping for balance, choose weights that achieve balance exactly and stay as close to uniform as possible. Entropy balancing solves a convex problem whose solution is exponential tilting, implies a logistic propensity score, and is doubly robust for linear outcome models.

**Learning outcomes**

- Formulate entropy balancing as minimum-KL weighting subject to moment constraints.
- Derive the exponential-tilting solution and the convex dual, and solve small examples.
- Prove the implied-logistic-propensity and double-robustness properties.
- Apply calibration weights to programme evaluation and survey raking, and recognise infeasible targets.

## Theory

### The problem

To estimate the ATT, reweight controls so their covariate moments match the treated group's. Let $c(X)$ be the balance functions (means, perhaps squares and interactions) and $\bar c_1$ their treated means. **Entropy balancing** (Hainmueller, 2012) solves

$$
\min_{w}\sum_{i: D_i = 0}w_i\ln\frac{w_i}{q_i} \quad \text{s.t.} \quad \sum_{i: D_i = 0}w_ic(X_i) = \bar c_1,\; \sum_{i: D_i = 0}w_i = 1,\; w_i \ge 0,
$$

with base weights $q_i$ (usually uniform). This minimises $D_{KL}(w \,\|\, q)$ (Day 83) subject to exact balance.

### Solution: exponential tilting

The Lagrangian gives

$$
w_i = \frac{q_i\exp(\lambda^\top c(X_i))}{\sum_j q_j\exp(\lambda^\top c(X_j))},
$$

where $\lambda$ minimises the convex **dual** $L(\lambda) = \ln\sum_j q_j\exp\big(\lambda^\top(c(X_j) - \bar c_1)\big)$. Its gradient is the weighted mean of $c(X)$ minus the target, so Newton's method converges quickly.

### Properties

- **Exact balance** on the chosen moments, by construction.
- **Implied propensity score:** ATT weights should be $e/(1 - e)$; exponential weights correspond to $e/(1 - e) = \exp(\alpha + \lambda^\top c(x))$, a logistic model in $c(X)$.
- **Double robustness:** the weighted control mean of $Y$ is consistent for $E[Y(0) \mid D = 1]$ if either the outcome model $E[Y(0) \mid X]$ is linear in $c(X)$, or the logistic propensity model in $c(X)$ is correct (Zhao and Percival).
- **Feasibility:** if the treated means lie outside the convex hull of control values, no non-negative weights exist (a sign of poor overlap).

### Family resemblance

Survey **calibration** and **raking** (Deville and Särndal) adjust weights to match known population totals with the same logic; the **covariate balancing propensity score (CBPS)** estimates a logistic score whose implied weights balance covariates; **stable balancing weights** minimise variance instead of KL, allowing approximate balance.

## Part 1 · Warm-up

### Exercise 1 · Checking a constraint

Control ages 30, 40, 50 with weights 0.2, 0.3, 0.5. Do they match a treated mean age of 43?

<details><summary>Show solution</summary>

**Step 1.** Weighted mean: $0.2(30) + 0.3(40) + 0.5(50) = 6 + 12 + 25 = 43$.

**Answer:** yes.

</details>

*Your work:*

### Exercise 2 · A one-covariate solution

Controls have $x = 0, 1, 2$ with equal base weights; the treated mean is 1.4. Find the entropy-balancing weights.

<details><summary>Show solution</summary>

**Step 1.** $w_k \propto t^{x_k}$ with $t = e^\lambda$: weights $1, t, t^2$ over $1 + t + t^2$.

**Step 2.** Constraint: $(t + 2t^2)/(1 + t + t^2) = 1.4$, so $0.6t^2 - 0.4t - 1.4 = 0$.

**Step 3.** $t = (0.4 + \sqrt{0.16 + 3.36})/1.2 \approx 1.897$, $\lambda \approx 0.640$.

**Step 4.** Weights: $1, 1.897, 3.598$ over 6.495: 0.154, 0.292, 0.554. Check: $0.292 + 2(0.554) = 1.40$.

</details>

*Your work:*

### Exercise 3 · Infeasible targets

In Exercise 2, could the target mean be 2.5?

<details><summary>Show solution</summary>

**Step 1.** Any weighted mean of 0, 1, 2 with non-negative weights lies in $[0, 2]$.

**Answer:** no: the treated units are outside the controls' support; no reweighting can fix that.

</details>

*Your work:*

### Exercise 4 · KL of the solution

Compute $D_{KL}(w \,\|\, u)$ for Exercise 2's weights.

<details><summary>Show solution</summary>

**Step 1.** $\sum w\ln(3w) = 0.154\ln 0.462 + 0.292\ln 0.876 + 0.554\ln 1.662$.

**Step 2.** $= -0.119 - 0.039 + 0.282 \approx 0.124$.

</details>

*Your work:*

### Exercise 5 · Why not just IPW?

Give two advantages of entropy balancing over logistic IPW for the ATT.

<details><summary>Show solution</summary>

**Step 1.** Balance on the chosen moments is exact, not hoped for; no iterating between model fitting and balance checks.

**Step 2.** Weights are as close to uniform as possible given balance (less variance than many IPW fits), and the estimator is doubly robust for linear outcome models.

</details>

## Part 2 · Proofs

*Your work:*

### Exercise 6 · Exponential tilting

Derive the solution form from the Lagrangian.

<details><summary>Show solution</summary>

**Step 1.** $\mathcal L = \sum_i w_i\ln(w_i/q_i) - \lambda^\top\big(\sum_i w_ic_i - \bar c_1\big) - \nu\big(\sum_i w_i - 1\big)$.

**Step 2.** $\partial\mathcal L/\partial w_i = \ln(w_i/q_i) + 1 - \lambda^\top c_i - \nu = 0$.

**Step 3.** $w_i = q_i\exp(\lambda^\top c_i + \nu - 1)$; normalising removes $\nu$. Non-negativity holds automatically. $\blacksquare$

</details>

*Your work:*

### Exercise 7 · The dual and its gradient

Show that $L(\lambda) = \ln\sum_j q_j\exp(\lambda^\top(c_j - \bar c_1))$ is convex with gradient $\sum_j w_j(\lambda)c_j - \bar c_1$.

<details><summary>Show solution</summary>

**Step 1.** Differentiate: $\nabla L = \sum_j q_je^{\lambda^\top(c_j - \bar c_1)}(c_j - \bar c_1)/\sum_k q_ke^{\lambda^\top(c_k - \bar c_1)} = \sum_j w_j(c_j - \bar c_1)$.

**Step 2.** The Hessian is the weighted covariance matrix of $c_j$, positive semi-definite: $L$ is convex (log-sum-exp).

**Step 3.** Setting the gradient to zero is exactly the balance constraint. $\blacksquare$

</details>

*Your work:*

### Exercise 8 · The implied propensity score

Show that if entropy-balancing weights equal the ATT odds $e(X)/(1 - e(X))$ up to a constant, then $e(X)$ is logistic in $c(X)$.

<details><summary>Show solution</summary>

**Step 1.** $e/(1 - e) = k\exp(\lambda^\top c(X))$ for some $k > 0$.

**Step 2.** Taking logs: $\ln(e/(1 - e)) = \ln k + \lambda^\top c(X)$, a logistic model. $\blacksquare$

</details>

*Your work:*

### Exercise 9 · Double robustness (outcome side)

Assume $E[Y(0) \mid X] = \beta^\top c(X)$ (with a constant in $c$). Show the entropy-balanced control mean of $Y$ is unbiased for $\beta^\top\bar c_1$, the treated units' average untreated outcome, conditional on the covariates.

<details><summary>Show solution</summary>

**Step 1.** $E\big[\sum_{D_i = 0}w_iY_i \mid X\big] = \sum_{D_i = 0}w_i\beta^\top c(X_i)$, since weights depend only on covariates.

**Step 2.** By exact balance, $= \beta^\top\sum w_ic(X_i) = \beta^\top\bar c_1$.

**Step 3.** $\beta^\top\bar c_1 = \frac{1}{N_1}\sum_{D_i = 1}E[Y_i(0) \mid X_i]$. $\blacksquare$ No propensity model is needed.

</details>

*Your work:*

### Exercise 10 · Balancing squares

Show that if $c(X)$ includes $X$ and $X^2$, the weighted control variance of $X$ equals the treated variance.

<details><summary>Show solution</summary>

**Step 1.** Exact balance on $X$ and $X^2$: $\sum w_iX_i = \bar x_1$ and $\sum w_iX_i^2 = \overline{x^2}_1$.

**Step 2.** Weighted variance $= \sum w_iX_i^2 - (\sum w_iX_i)^2 = \overline{x^2}_1 - \bar x_1^2$, the treated (population-style) variance. $\blacksquare$

</details>

## Part 3 · Applied

*Your work:*

### Exercise 11 · A training programme

In the simulation below, trainees differ from non-trainees in age, schooling and prior earnings; the true ATT is \$1,800. Compare the naive difference, logistic IPW (ATT weights) and entropy balancing on means.

<details><summary>Show solution</summary>

**Step 1.** Naive: about −\$3,500 (trainees were poorer).

**Step 2.** Logistic IPW: about \$2,000, off by roughly \$200, with an SD of about \$280 across simulations; the fitted weights leave some imbalance.

**Step 3.** Entropy balancing: exact mean balance, mean estimate about \$1,800 and an SD of about \$180.

</details>

*Your work:*

### Exercise 12 · Raking a health survey

A survey has cell shares (male young, male old, female young, female old) = (0.3, 0.2, 0.3, 0.2). Census margins: 48% male, 40% young. Carry out one raking iteration (sex, then age).

<details><summary>Show solution</summary>

**Step 1.** Sex: male share 0.5 → factor 0.96; female 0.5 → factor 1.04. Cells: 0.288, 0.192, 0.312, 0.208.

**Step 2.** Age: young share 0.6 → factor 0.667; old 0.4 → factor 1.5. Cells: 0.192, 0.288, 0.208, 0.312.

**Step 3.** Check: male $0.192 + 0.288 = 0.48$; young $0.192 + 0.208 = 0.40$.

**Answer:** both margins match after one iteration here (in general, iterate until convergence).

</details>

*Your work:*

### Exercise 13 · Infeasible balance in hospital comparisons

Entropy balancing fails to converge when weighting community-hospital patients to match a cancer centre's patients on tumour stage. What does this reveal and what can be done?

<details><summary>Show solution</summary>

**Step 1.** The cancer centre's stage distribution lies outside what community hospitals' patients can reproduce (e.g., no stage-IV patients there).

**Step 2.** Options: restrict to the overlapping population (stages present in both), relax to approximate balance, or report that the comparison is not supported by the data.

</details>

*Your work:*

### Exercise 14 · Balancing more than means

Why add squared income and an indicator for income below the poverty line to the balance functions in a study of cash transfers and child nutrition?

<details><summary>Show solution</summary>

**Step 1.** Nutrition depends non-linearly on income, particularly at the bottom.

**Step 2.** Balancing those functions makes the outcome model "linear in $c(X)$" more plausible, invoking double robustness (Exercise 9).

**Answer:** more balance functions protect against non-linearity; the cost is more extreme weights (check ESS).

</details>

*Your work:*

### Exercise 15 · Reporting weighting results

What should a paper using entropy balancing report?

<details><summary>Show solution</summary>

**Step 1.** The balance functions chosen and why.

**Step 2.** Balance before and after (exact on chosen moments; also report unbalanced features such as other moments).

**Step 3.** Weight diagnostics: maximum weight, ESS, KL from uniform.

**Step 4.** Estimates with standard errors that account for weight estimation (bootstrap or M-estimation), and sensitivity analysis.

</details>

## Verify in Python

*Your work:*

In [ ]:
import numpy as np

def ebal(C, target, iters=50):
    """Entropy-balancing weights for rows of C to match target means (Newton on the dual)."""
    lam = np.zeros(C.shape[1]); Cc = C - target
    for _ in range(iters):
        a = Cc @ lam; w = np.exp(a - a.max()); w /= w.sum()
        g = w @ Cc; Hm = (Cc * w[:, None]).T @ Cc - np.outer(g, g)
        lam -= np.linalg.solve(Hm + 1e-10 * np.eye(len(lam)), g)
    return w

w = ebal(np.array([[0.], [1.], [2.]]), np.array([1.4]))
print("Ex 2 weights:", w.round(3), " mean:", round(w @ [0, 1, 2], 4), " KL:", round((w * np.log(3 * w)).sum(), 3))
print("Ex 12 raking:", (np.array([.3, .2, .3, .2]) * np.array([.96, .96, 1.04, 1.04]) * np.array([2 / 3, 1.5, 2 / 3, 1.5])).round(3))

# Exercise 11
rng = np.random.default_rng(84)
def logit_fit(X, d, iters=25):
    b = np.zeros(X.shape[1])
    for _ in range(iters):
        p = 1 / (1 + np.exp(-X @ b)); b += np.linalg.solve(X.T @ ((p * (1 - p))[:, None] * X), X.T @ (d - p))
    return 1 / (1 + np.exp(-X @ b))
res = {"naive": [], "logistic IPW": [], "entropy balancing": []}
for _ in range(200):
    n1, n0 = 300, 3000
    age = np.r_[rng.normal(25, 5, n1), rng.normal(33, 8, n0)]; edu = np.r_[rng.normal(10, 1.5, n1), rng.normal(11.5, 2, n0)]
    pre = np.r_[rng.gamma(2, 1500, n1), rng.gamma(3, 2500, n0)] / 1000
    D = np.r_[np.ones(n1), np.zeros(n0)].astype(bool)
    Y = 2000 + 120 * age + 500 * edu + 600 * pre + 15 * pre ** 2 + rng.normal(0, 2000, n1 + n0) + 1800 * D
    X = np.column_stack([age, edu, pre])
    e = logit_fit(np.column_stack([np.ones(n1 + n0), X]), D.astype(float)); wi = (e / (1 - e))[~D]
    we = ebal(X[~D], X[D].mean(0))
    res["naive"].append(Y[D].mean() - Y[~D].mean())
    res["logistic IPW"].append(Y[D].mean() - wi @ Y[~D] / wi.sum())
    res["entropy balancing"].append(Y[D].mean() - we @ Y[~D])
for k, v in res.items():
    print(f"{k:18s} mean {np.mean(v):8,.0f}   SD {np.std(v):6,.0f}   (truth 1,800)")

## Key takeaways

- Entropy balancing finds the weights closest to uniform (in KL) that balance chosen covariate moments exactly.
- The solution is exponential tilting, found by minimising a convex dual; it implies a logistic propensity score.
- It is doubly robust for outcome models linear in the balance functions; infeasibility signals poor overlap.